# Setup
Inputs stay in this ensemble. Generated files go to its ignored project workspace.


In [1]:
"""A24 light-current three-point analysis and paper plots."""

from pathlib import Path
import os
import sys

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd().resolve()
if HERE.parent.name == "__codex_ignore":
    HERE = HERE.parent.parent / HERE.name
if HERE.name != "cA211.530.24" or HERE.parent.name != "07_Nsgm":
    raise RuntimeError("Launch this notebook from its cA211.530.24 directory.")
WORK = HERE.parent / "__codex_ignore" / HERE.name
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(HERE.parent))

import util as yu
import util_codex as yuc

yu.setpath("analysis_3pt_light_codex")
CONFIG = {
    "ensemble": "a24", "tfs": [10, 12, 14], "lower_bounds": [10, 12, 14],
    "ylim": (80, 240), "yticks": [80, 120, 160, 200, 240],
    "standard_selection": None, "gevp_selection": None,
    "laplace_selection": (12, 4), "rlg_selection": (12, 4),
}
ENS = CONFIG["ensemble"]
DISPLACEMENT = 2
LAPLACE_CUT = 4
yuc.apply_paper_style()


# Analysis
## Input correlators


In [2]:
TFS = CONFIG["tfs"]
c2pt_matrix, tf2c3pt_matrix, c2pt_nucleon = yu.load_pkl(HERE / "pkl/processData/reg_ignore/data.pkl")
tf2c3pt_matrix = {tf: np.real(tf2c3pt_matrix[tf]) for tf in TFS}
for tf, values in tf2c3pt_matrix.items():
    if values.shape != (len(c2pt_matrix), tf + 1, 2, 2):
        raise ValueError(f"Unexpected three-point sample/time axes at t_s/a={tf}")
    if np.shape(c2pt_nucleon[tf]) != (len(c2pt_matrix),):
        raise ValueError(f"Two- and three-point samples differ at t_s/a={tf}")
c2pt = np.real(c2pt_matrix)


## GEVP eigenvectors and weight


In [3]:
two_point = yu.load_pkl(WORK / "pkl/analysis_2pt_codex/reg_ignore/two_point_references.pkl")
v, w = (two_point["weights"][name] for name in ["v", "w"])
assert len(v) == len(w) == len(c2pt)
GEVP_TAG = yuc.sample_cache_tag(c2pt, v, w, *[tf2c3pt_matrix[t] for t in TFS])
print("GEVP input:", two_point["gevp_delta"], two_point["gevp_window"],
      "v", yu.jackme_un2str(v), "W", yu.jackme_un2str(w))


GEVP input: 4 (11, 14) v 0.121(15) W 0.0473(67)


## Standard and reduced-GEVP ratios


In [4]:
ratios = {key: {} for key in ["standard", "gevp"]}
correlators = {key: {} for key in ["c3_standard", "c2_standard", "c3_gevp", "c2_gevp"]}
for tf in TFS:
    c3pt = tf2c3pt_matrix[tf]
    c3_standard = c3pt[:, :, 0, 0]
    c2_standard = np.real(c2pt_nucleon[tf])

    v_column, w_column = v[:, None], w[:, None]
    c3_gevp = (1 - w_column**2) * c3_standard
    c3_gevp += v_column * (1 + w_column) * (c3pt[:, :, 0, 1] + c3pt[:, :, 1, 0])
    c2_gevp = c2_standard + v * (c2pt[:, tf, 0, 1] + c2pt[:, tf, 1, 0])
    c2_gevp += v**2 * c2pt[:, tf, 1, 1]

    ratios["standard"][tf] = c3_standard / c2_standard[:, None]
    ratios["gevp"][tf] = c3_gevp / c2_gevp[:, None]
    for key, value in zip(correlators, [c3_standard, c2_standard, c3_gevp, c2_gevp]):
        correlators[key][tf] = value


## Nucleon two-point input
Read the selected fit from the two-point notebook instead of fitting it again.


In [5]:
nucleon_fit = two_point["nucleon"]
assert len(nucleon_fit) == len(c2pt)
print(f"{ENS}: nucleon input from analysis_2pt_codex, "
      f"mN={yu.jackme_un2str(nucleon_fit[:, 0] * yu.ens2aInv[ENS])} MeV")

yuc.guard_fit_cache(nucleon_fit, v, w, *ratios["standard"].values(), *ratios["gevp"].values())


a24: nucleon input from analysis_2pt_codex, mN=1209.0(2.8) MeV


## Standard-ratio cases I-III and constant-fit diagnostics


In [6]:
lower_bounds = CONFIG["lower_bounds"]
standard = yu.doFits_3pt(
    "2st2step_SYMshare", ratios["standard"], lower_bounds[:-1], [2],
    pars_jk_meff2st=nucleon_fit, symmetrizeQ=True,
    label=f"{ENS}_standard_shared_two_state_codex", overwrite=False,
)
standard_models = {"standard_I": standard}
for case, model in [("II", "2st2step_SYM"), ("III", "2st2step_SYM_share11")]:
    standard_models["standard_" + case] = []
    for lower in lower_bounds[:-1]:
        excluded = CONFIG.get("excluded_fits", {}).get("standard_" + case, [])
        if (lower, 2) in excluded:
            continue
        print(f"{ENS}: standard case {case}, lower={lower}", flush=True)
        fit = yu.doFits_3pt(
            model, ratios["standard"], [lower], [2], pars_jk_meff2st=nucleon_fit,
            symmetrizeQ=True, label=f"{ENS}_standard_{case}_tf{lower}_codex", overwrite=False,
        )
        standard_models["standard_" + case].extend(fit)



a24: standard case II, lower=10


a24: standard case II, lower=12


a24: standard case III, lower=10


a24: standard case III, lower=12


## Laplace filter and fits


In [7]:
def laplace_ratio(tf2c3pt, tf2c2pt, energy, displacement=DISPLACEMENT):
    energy = np.asarray(energy)
    eigenvalue = 2 * np.cosh(displacement * energy) - 2
    c3_eigenvalue = eigenvalue if energy.ndim == 0 else eigenvalue[:, None]
    ratios = {}
    for tf, c3pt in tf2c3pt.items():
        filtered = -(np.roll(c3pt, -displacement, 1) + np.roll(c3pt, displacement, 1) - 2 * c3pt)
        ratios[tf] = (filtered + c3_eigenvalue * c3pt) / (eigenvalue * tf2c2pt[tf])[:, None]
    return ratios

first_tf = CONFIG["tfs"][0]
initial_ground = np.mean(ratios["standard"][first_tf][:, first_tf // 2])
laplace = yu.doFits_3pt_lbd(
    lambda energy: laplace_ratio(correlators["c3_standard"], correlators["c2_standard"], energy),
    lower_bounds[:-1], [3, 4], pars0=[initial_ground, .16], symmetrizeQ=True,
    label=f"{ENS}_standard_laplace_delta2_codex", overwrite=False,
)
laplace += yu.doFits_3pt_lbd(
    lambda energy: laplace_ratio(correlators["c3_standard"], correlators["c2_standard"], energy),
    lower_bounds[-1:], [4], pars0=[initial_ground, .16], symmetrizeQ=True,
    label=f"{ENS}_standard_laplace_delta2_endpoint_codex", overwrite=False,
)
laplace = [[label, np.column_stack([pars[:, 0], np.abs(pars[:, 1])]), chi2, ndof]
           for label, pars, chi2, ndof in laplace]
selected_laplace = next(fit for fit in laplace if fit[0] == CONFIG["laplace_selection"])
energy = selected_laplace[1][:, 1]
ratios["laplace"] = laplace_ratio(correlators["c3_standard"], correlators["c2_standard"], energy)
ratios["rlg"] = laplace_ratio(correlators["c3_gevp"], correlators["c2_gevp"], energy)
fits = {"standard": standard, "laplace": laplace, **standard_models}


## Case-V fits


In [8]:
import warnings

fits["standard_V"] = yu.doFits_3pt(
    "2st2step_SYM_0rc1_0ra11", ratios["standard"], lower_bounds, [2],
    symmetrizeQ=True, label=f"{ENS}_standard_caseIV_all_bounds_codex", overwrite=False)

fits["rlg_V"], rlg_fit_status = [], {}
for lower in lower_bounds:
    label = f"{ENS}_rlg_IV_lower{lower}_{GEVP_TAG}"
    fit = yu.load_pkl_internal(label)
    if fit is None:
        with warnings.catch_warnings(record=True):
            fit = yu.doFit_3pt("2st2step_SYM_0rc1_0ra11", ratios["rlg"], lower, LAPLACE_CUT,
                               symmetrizeQ=True)
        yu.save_pkl_internal(label, fit)
    if fit is None:
        rlg_fit_status[lower] = "insufficient data"
        continue
    pars, chi2, ndof, warning_count = fit
    valid = warning_count == 0 and np.isfinite(pars).all() and np.isfinite(chi2).all()
    sigma_error = yu.jackme(pars[:, 0])[1] * yu.ens2amul[ENS] * yu.ens2aInv[ENS]
    if ENS == "a":
        sigma_error *= (135 / yu.ens2mpi[ENS])**2
    resolved = np.mean(pars[:, 1]) > 0 and sigma_error < np.ptp(CONFIG["ylim"])
    rlg_fit_status[lower] = dict(converged=bool(valid), displayed=bool(valid and resolved),
                                warnings=int(warning_count))
    if valid and resolved:
        fits["rlg_V"].append([(lower, LAPLACE_CUT), pars, chi2, ndof])
    print("RLG V", lower, rlg_fit_status[lower], flush=True)
yu.save_pkl_reg("rlg_fit_status", rlg_fit_status)

for name, cut in [("standard", 2), ("rlg", LAPLACE_CUT)]:
    fits[name + "_IV"] = []
    tag = yuc.sample_cache_tag(nucleon_fit, *ratios[name].values())
    for lower in lower_bounds:
        cache = f"{ENS}_{name}_IV_zero_r11_{lower}_{cut}_{tag}"
        fit = yu.load_pkl_internal(cache)
        if fit is None:
            previous = next((f for f in fits[name + "_V"] if f[0] == (lower, cut)), None)
            seed = None if previous is None else np.mean(previous[1], axis=0).tolist()
            print("Fit", name, "IV", lower, flush=True)
            with warnings.catch_warnings(record=True):
                fit = yu.doFit_3pt(
                    "2st2step_SYM_0ra11", ratios[name], lower, cut,
                    pars_jk_meff2st=nucleon_fit, pars0=seed, symmetrizeQ=True)
            yu.save_pkl_internal(cache, fit)
        if fit is None:
            continue
        pars, chi2, ndof, warning_count = fit
        valid = warning_count == 0 and np.isfinite(pars).all() and np.isfinite(chi2).all()
        resolved = np.mean(pars[:, 1]) > 0
        if name == "rlg":
            scale = yu.ens2amul[ENS] * yu.ens2aInv[ENS]
            if ENS == "a":
                scale *= (135 / yu.ens2mpi[ENS])**2
            resolved &= yu.jackme(pars[:, 0])[1] * scale < np.ptp(CONFIG["ylim"])
        if valid and resolved:
            fits[name + "_IV"].append([(lower, cut), pars, chi2, ndof])
        print(name, "IV", lower, "warnings", warning_count, "displayed", bool(valid and resolved),
              "gap", yu.jackme_un2str(pars[:, 1] * yu.ens2aInv[ENS]), flush=True)


RLG V 10 {'converged': True, 'displayed': True, 'warnings': 0}


RLG V 12 {'converged': True, 'displayed': True, 'warnings': 0}


RLG V 14 {'converged': True, 'displayed': True, 'warnings': 0}


standard IV 10 warnings 0 displayed True gap 570(50)


standard IV 12 warnings 0 displayed True gap 597(78)


standard IV 14 warnings 0 displayed True gap 600(130)


rlg IV 10 warnings 0 displayed True gap 930(430)


rlg IV 12 warnings 0 displayed True gap 1200(1200)


rlg IV 14 warnings 0 displayed True gap 1400(2600)


## Physical units and fit summaries


In [9]:
def print_fits(ensemble, name, fits, yunit, energy=False):
    print(f"\n{ensemble}: {name}")
    for label, parameters, chi2, ndof in fits:
        gap = f", dE={yu.jackme_un2str(parameters[:, 1] * yu.ens2aInv[ensemble])} MeV" if energy else ""
        probability = yu.chi2Ndof2pval(np.mean(chi2), ndof)
        print(f"  {label}: sigma_piN={yu.jackme_un2str(parameters[:, 0] * yunit)} MeV{gap}, p={probability:.3f}")

yunit = yu.ens2amul[CONFIG["ensemble"]] * yu.ens2aInv[CONFIG["ensemble"]]
result = {"ratios": ratios, "fits": fits, "yunit": yunit}

for name, energy in [("standard", False), ("standard_II", True), ("standard_III", True),
                     ("standard_IV", True), ("standard_V", True), ("laplace", True), ("rlg_IV", True), ("rlg_V", True)]:
    print_fits(CONFIG["ensemble"], name, fits[name], yunit, energy)



a24: standard
  (10, 2): sigma_piN=162.1(6.6) MeV, p=0.000
  (12, 2): sigma_piN=172.5(8.4) MeV, p=0.018

a24: standard_II
  (10, 2): sigma_piN=199(12) MeV, dE=545(58) MeV, p=0.553
  (12, 2): sigma_piN=196(13) MeV, dE=602(78) MeV, p=0.797

a24: standard_III
  (10, 2): sigma_piN=201(13) MeV, dE=543(58) MeV, p=0.569
  (12, 2): sigma_piN=195(12) MeV, dE=602(78) MeV, p=0.797

a24: standard_IV
  (10, 2): sigma_piN=200(10) MeV, dE=570(50) MeV, p=0.509
  (12, 2): sigma_piN=194(12) MeV, dE=597(78) MeV, p=0.863
  (14, 2): sigma_piN=193(17) MeV, dE=600(130) MeV, p=0.392

a24: standard_V
  (10, 2): sigma_piN=201(11) MeV, dE=558(50) MeV, p=0.524
  (12, 2): sigma_piN=194(13) MeV, dE=593(79) MeV, p=0.865
  (14, 2): sigma_piN=193(17) MeV, dE=600(130) MeV, p=0.392

a24: laplace
  (10, 3): sigma_piN=168.6(4.2) MeV, dE=832(45) MeV, p=0.000
  (10, 4): sigma_piN=199(13) MeV, dE=562(62) MeV, p=0.842
  (12, 3): sigma_piN=178.6(5.5) MeV, dE=738(49) MeV, p=0.593
  (12, 4): sigma_piN=193(13) MeV, dE=602(87) Me

## Energy-comparison inputs
Use the current case-IV fits and paired two-point references.


In [10]:
XUNIT, YUNIT, AINV_GEV = yu.ens2a[ENS], yunit, yu.ens2aInv[ENS] / 1000
nucleon_standard, nsigma_selected = two_point["nucleon"], two_point["nsigma"]
selected_lower = CONFIG["laplace_selection"][0]
standard_gap = next(f[1][:, 1] for f in fits["standard_IV"] if f[0] == (selected_lower, 2))
laplace_gap = selected_laplace[1][:, 1]
# Use the same RLG lower bound as B64 for this energy comparison.
selected_rlg = next(f for f in fits["rlg_IV"] if f[0] == (10, LAPLACE_CUT))
rlg_gap = selected_rlg[1][:, 1]


# Plotting
## Ratio and fit-scan helpers


In [11]:
def plot_config(config):
    a = yu.ens2a[config["ensemble"]]
    extent = max(config["tfs"]) / 2 * a
    time_limits = (np.array([min(config["tfs"]), max(config["tfs"])]) + [-.8, .8]) * a
    return {"limits": dict(ylim=config["ylim"], yticks=config["yticks"]),
            "selection": (config["laplace_selection"][0], 2), "selection_case": "IV",
            "rainbow": dict(xlim=(-extent, extent), xticks=np.arange(-.6, .61, .3)),
            "midpoint": dict(xlim=time_limits, xticks=np.arange(1, time_limits[1], .2)),
            "scan": dict(xlim=time_limits, xticks=np.arange(1, time_limits[1], .2))}


## Standard and transformed comparison


In [12]:
def make_standard_comparison(result, config, output_name):
    ensemble = config["ensemble"]
    display = plot_config(config)
    display["limits"] = dict(ylim=(80, 280), yticks=[80, 120, 160, 200, 240, 280])
    yuc.plot_standard_gevp(result["ratios"]["standard"], result["ratios"]["gevp"],
        [result["fits"]["standard_" + case] for case in ["I", "II", "III", "IV", "V"]],
        nucleon_fit, two_point["nsigma"], yu.ens2a[ensemble], result["yunit"],
        yu.ens2aInv[ensemble] / 1000, display, output_name)


## Laplace comparison


In [13]:
def make_laplace_summary(result, config, output_name):
    ensemble = config["ensemble"]
    yuc.plot_laplace_summary(*[result["ratios"][key] for key in ["standard", "laplace", "gevp", "rlg"]],
        [result["fits"][key] for key in ["standard_V", "laplace", "rlg_V"]],
        yu.ens2a[ensemble], result["yunit"], yu.ens2aInv[ensemble] / 1000, plot_config(config), output_name,
        fits_laplace_gevp_iv=result["fits"]["rlg_IV"])


## Energy comparison


In [14]:
def plot_energy_scales():
    yuc.plot_energy_scales(ENS, nucleon_standard, nsigma_selected, standard_gap,
                          laplace_gap, rlg_gap, AINV_GEV, show_resonances=False)


## Generate figures


In [15]:
make_standard_comparison(result, CONFIG, "Rstd_RGEVP_light_A24")
make_laplace_summary(result, CONFIG, "Laplace_summary_light_A24")
with mpl.rc_context(yuc.paper_style({"lines.markersize": 3.3, "errorbar.capsize": 2.5})):
    plot_energy_scales()
